In [0]:
%python
import os, random
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

dbutils.widgets.text("run_date", "")
run_date = dbutils.widgets.get("run_date")

# The job passes today's date; we create the file for YESTERDAY
today     = date.fromisoformat(run_date) if run_date else datetime.now(ZoneInfo("America/New_York")).date()
data_date = today - timedelta(days=1)

raw_dir   = "/Volumes/workspace/default/raw"
seed_file = f"{raw_dir}/neo4j_query_table_data_2026-10-28.csv"   # your real file, used as a template
out_file  = f"{raw_dir}/neo4j_query_table_data_{data_date.isoformat()}.csv"

# Safe to re-run: if yesterday's file already exists, do nothing
if os.path.exists(out_file):
    dbutils.notebook.exit(f"Skipped: {out_file} already exists")

print(f"Will create: {out_file}")

In [0]:
%python
seed = (spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(seed_file)
        .toPandas())

n   = random.randint(15, 40)                                    # 15–40 new jobs per day
new = seed.sample(n=n, replace=True).reset_index(drop=True)     # real companies, titles, descriptions

locations     = seed["location"].dropna().unique().tolist()
new["job_id"]   = [str(random.randint(4_000_000_000, 4_999_999_999)) for _ in range(n)]  # fake IDs
new["location"] = [random.choice(locations) for _ in range(n)]                          # shuffled locations
new["url"]      = "https://www.linkedin.com/jobs/view/" + new["job_id"]                 # fake URLs

new.to_csv(out_file, index=False)
print(f"Created {out_file} with {n} rows")
print(new[["job_id", "company", "title", "location"]].head(10).to_string(index=False))

Created /Volumes/workspace/default/raw/neo4j_query_table_data_2026-10-27.csv with 33 rows
    job_id                   company                                             title                        location
4223191840                 ROCKCREST               Power BI Visual Creator Consultant              Ridgefield Park, NJ
4173006503     Zodiac Solutions, Inc                                Power BI Developer         Colorado, United States
4783741171                Sumeru Inc                                Lead Data Engineer                   Arlington, VA
4862386582               Onward Play                    Data Engineer - Python [74476] Austin, Texas Metropolitan Area
4414623584              Apex Systems                                  Sr Data Engineer                     Oshkosh, WI
4706411268                   Verizon Social Media Business Intelligence Senior Manager        Tennessee, United States
4519871997                IntelyCare                            Data Platform